In [72]:
import pandas as pd
import numpy as np

fact_orders = pd.read_csv("../data_raw/OrderList.csv")
dim_freight = pd.read_csv("../data_raw/FreightRates.csv")
dim_ports = pd.read_csv("../data_raw/PlantPorts.csv")
dim_products = pd.read_csv("../data_raw/ProductsPerPlant.csv")
dim_customers = pd.read_csv("../data_raw/VmiCustomers.csv")
dim_wh_capacity = pd.read_csv("../data_raw/WhCapacities.csv")
dim_wh_cost = pd.read_csv("../data_raw/WhCosts.csv") 

# fact_orders.head()
# fact_orders.info()
# fact_orders.shape
# fact_orders.isnull().sum()
# fact_orders.duplicated().sum()


fact_orders.columns = (
    fact_orders.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# fact_orders.head()

fact_orders['order_date'] = pd.to_datetime(fact_orders['order_date'])

# fact_orders.info()

fact_orders['total_weight'] = fact_orders['unit_quant'] * fact_orders['weight']
# fact_orders['total_weight']

fact_orders['late_delivery'] = np.where(
    fact_orders['ship_late_day_count'] > 0, 1, 0
)

fact_orders['actual_transit_days'] = (
    fact_orders['tpt_day_count']
    + fact_orders['ship_late_day_count']
    - fact_orders['ship_ahead_day_count']
)

fact_orders['year'] = fact_orders['order_date'].dt.year
fact_orders['month'] = fact_orders['order_date'].dt.month

fact_orders.head()

fact_orders['late_delivery'].value_counts()

fact_orders.groupby('carrier')['late_delivery'].mean().sort_values(ascending=False)

fact_orders.groupby('service_level')['late_delivery'].mean().sort_values(ascending=False)

fact_orders.groupby('orig_port')['late_delivery'].mean().sort_values(ascending=False)

fact_orders.groupby(['year','month'])['late_delivery'].mean()

fact_orders['service_level'].value_counts()

fact_orders.groupby('service_level')['late_delivery'].mean().sort_values(ascending=False)

#sql parl

dim_freight.head()
dim_products.head()
dim_wh_capacity.head()
dim_wh_cost.head()

dim_freight.columns
dim_products.columns
dim_wh_capacity.columns
dim_wh_cost.columns

print("FreightRates Columns:", dim_freight.columns)
print("ProductsPerPlant Columns:", dim_products.columns)
print("WhCapacities Columns:", dim_wh_capacity.columns)
print("VmiCustomers Columns:", dim_customers.columns)

dim_freight.columns = dim_freight.columns.str.lower().str.strip()
dim_products.columns = dim_products.columns.str.lower().str.strip()
dim_wh_capacity.columns = dim_wh_capacity.columns.str.lower().str.strip()
dim_customers.columns = dim_customers.columns.str.lower().str.strip()
dim_wh_cost.columns = dim_wh_cost.columns.str.lower().str.strip()

#step 1

merged = fact_orders.merge(
    dim_freight,
    on=['carrier','orig_port','dest_port','service_level','tpt_day_count'],
    how='left'
)

#step 2

merged = merged[
    (merged['weight'] >= merged['min_weight_quant']) &
    (merged['weight'] <= merged['max_weight_quant'])
]

#step 3

merged['freight_cost'] = merged['rate'] * merged['unit_quant']

#imp step

merged.shape
merged['freight_cost'].isnull().sum()

#phase 8 - calculating warehouse handling cost

#step 1- merging warehouse cost

merged = merged.merge(
    dim_wh_cost,
    on='plant_code',
    how='left'
)

#step 2 - caluclating handling cost

merged['handling_cost'] = merged['cost_per_unit'] * merged['unit_quant']
merged['handling_cost']

#step 3 - calculating total shipment cost

merged['total_shipment_cost'] = (
    merged['freight_cost'] + merged['handling_cost']
)
merged['total_shipment_cost']

#checking

merged[['freight_cost','handling_cost','total_shipment_cost']].head()
merged['total_shipment_cost'].isnull().sum()

#phase 9 - Plant utilization using kpi (merging warehouse capacity)

#step 1 - merge capacity

merged = merged.merge(
    dim_wh_capacity,
    on='plant_code',
    how='left'
)

#step 2 calculating daily utilization

merged['capacity_utilization_pct'] = (
    merged['unit_quant'] / merged['daily_capacity']
) * 100
merged['capacity_utilization_pct']

# Phase 10 - Getting insigts from this

# top 5 plants after total shipment costs

merged.groupby('plant_code')['total_shipment_cost']\
      .sum()\
      .sort_values(ascending=False)\
      .head()

# now calculating average utlization of each plant

merged.groupby('plant_code')['capacity_utilization_pct']\
      .mean()\
      .sort_values(ascending=False)

# finding mopst expensive carrier

merged.groupby('carrier')['freight_cost']\
      .sum()\
      .sort_values(ascending=False)

# phase 10 - Build dataset that is useful in sql
# step 1 selecting columns

final_fact = merged[[
    'order_id',
    'order_date',
    'plant_code',
    'customer',
    'product_id',
    'carrier',
    'service_level',
    'orig_port',
    'dest_port',
    'unit_quant',
    'weight',
    'total_weight',
    'freight_cost',
    'handling_cost',
    'total_shipment_cost',
    'capacity_utilization_pct',
    'late_delivery',
    'actual_transit_days',
    'year',
    'month'
]]

final_fact.shape
final_fact.head()
# final_fact.info()

# Phase 11- Creating deminsion table
#step 1 dim for plants

dim_plants = merged[['plant_code','daily_capacity','cost_per_unit']].drop_duplicates()

#step 2 dim for carrier

dim_carrier = merged[['carrier','carrier_type','mode_dsc']].drop_duplicates()

#step 3 dim for service

dim_service = merged[['service_level']].drop_duplicates()

#step 4 dim for date

dim_date = final_fact[['order_date','year','month']].drop_duplicates()

# DATA CLEANED!

#phase 12 - Exporting the data set into data cleaned folder

final_fact.to_csv("../data_cleaned/fact_orders.csv", index=False)
dim_plants.to_csv("../data_cleaned/dim_plants.csv", index=False)
dim_carrier.to_csv("../data_cleaned/dim_carrier.csv", index=False)
dim_service.to_csv("../data_cleaned/dim_service.csv", index=False)
dim_date.to_csv("../data_cleaned/dim_date.csv", index=False)



FreightRates Columns: Index(['Carrier', 'Orig_Port', 'Dest_Port', 'Min_Weight_Quant',
       'Max_Weight_Quant', 'Service_Level', 'Min_Cost', 'Rate', 'Mode_DSC',
       'TPT_Day_Count', 'Carrier_Type'],
      dtype='str')
ProductsPerPlant Columns: Index(['Plant_Code', 'Product_ID'], dtype='str')
WhCapacities Columns: Index(['Plant_Code', 'Daily_Capacity'], dtype='str')
VmiCustomers Columns: Index(['Plant_Code', 'Customer'], dtype='str')
